# Challenge 02 Ready: Logistic regression

## Objective

This notebook is the fast-to-run version of the corresponding challenge model.
The hyperparameter search was already done previously, so here the model is configured with the best parameters found and is ready for manual execution.

Workflow:

1. Load and audit the data.
2. Explore key patterns graphically.
3. Split the data into training and validation subsets.
4. Train the model with fixed best hyperparameters.
5. Evaluate the model on the validation split.
6. Interpret the model.
7. Refit on the full training set and export a Kaggle-style submission.

## 0. Load libraries

In [ ]:
from pathlib import Path
import json
import warnings

import matplotlib.pyplot as plt
import nbformat
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.decomposition import PCA
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 12
plt.rcParams["savefig.bbox"] = "tight"

RANDOM_STATE = 301655

## 1. Configure the model with the fixed best hyperparameters

In [ ]:
from sklearn.linear_model import LogisticRegression

MODEL_NAME = "Logistic regression"
NOTEBOOK_SLUG = "challenge_02_logistic_regression_ready"

BEST_PARAMS = {
    "model__C": 1.0,
    "model__solver": "lbfgs",
}

MODEL_PIPELINE = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(C=1.0, solver="lbfgs", max_iter=4000, random_state=RANDOM_STATE)),
    ]
)

## 2. Load the challenge data

In [ ]:
BASE_DIR = Path.cwd()
TRAIN_PATH = BASE_DIR / "data" / "training.csv"
TEST_PATH = BASE_DIR / "data" / "test.csv"
SAMPLE_PATH = BASE_DIR / "data" / "sample.csv"

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)
sample_df = pd.read_csv(SAMPLE_PATH)

X = train_df.drop(columns=["id", "class"])
y = train_df["class"]
X_test_kaggle = test_df.drop(columns=["id"])

TEST_SIZE = 0.20

output_dir = BASE_DIR / "output" / NOTEBOOK_SLUG
output_dir.mkdir(parents=True, exist_ok=True)

submission_dir = BASE_DIR / "submissions"
submission_dir.mkdir(parents=True, exist_ok=True)

def save_current_figure(filename: str) -> Path:
    path = output_dir / filename
    plt.tight_layout()
    plt.savefig(path, dpi=200)
    return path

def evaluate_predictions(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    cm = confusion_matrix(y_true, y_pred)
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "confusion_matrix": cm.tolist(),
    }

## 3. First look at the training data

In [ ]:
train_df.head()

## 4. Check shape, target balance, and data types

In [ ]:
print("Training shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Sample submission shape:", sample_df.shape)
print("\nTarget distribution:")
display(train_df["class"].value_counts().sort_index())
print("\nDtypes summary:")
display(train_df.dtypes.value_counts())

## 5. Data quality audit

In [ ]:
quality_report = pd.DataFrame(
    {
        "missing_values": train_df.isna().sum(),
        "missing_pct": train_df.isna().mean().mul(100),
        "n_unique": train_df.nunique(),
    }
)

print("Duplicate rows in training:", int(train_df.duplicated().sum()))
print("Duplicated ids in training:", int(train_df["id"].duplicated().sum()))
print("Duplicated ids in test:", int(test_df["id"].duplicated().sum()))
print("Total missing values in training:", int(train_df.isna().sum().sum()))
print("Total missing values in test:", int(test_df.isna().sum().sum()))
quality_report.head(10)

## 6. Plot the class balance

In [ ]:
class_counts = train_df["class"].value_counts().sort_index()

plt.figure(figsize=(8, 5))
sns.barplot(x=class_counts.index.astype(str), y=class_counts.values, palette="viridis")
plt.title("Class balance in the training set")
plt.xlabel("Class")
plt.ylabel("Observations")
class_balance_path = save_current_figure("class_balance.png")
plt.show()

print(f"Saved figure: {class_balance_path}")

## 7. Identify features with the largest mean separation between classes

In [ ]:
feature_mean_gap = (
    train_df.groupby("class")
    .mean(numeric_only=True)
    .drop(columns=["id"])
    .diff()
    .iloc[-1]
    .abs()
    .sort_values(ascending=False)
)

top_gap_features = feature_mean_gap.head(15)
plt.figure(figsize=(12, 6))
sns.barplot(x=top_gap_features.values, y=top_gap_features.index, palette="mako")
plt.title("Top predictors by absolute difference in class means")
plt.xlabel("|mean(class=1) - mean(class=0)|")
plt.ylabel("Predictor")
gap_path = save_current_figure("top_mean_gaps.png")
plt.show()

print(f"Saved figure: {gap_path}")
display(top_gap_features.to_frame("absolute_mean_gap"))

## 8. Inspect the six most separated predictors

In [ ]:
top_boxplot_features = top_gap_features.head(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for axis, feature in zip(axes, top_boxplot_features):
    sns.boxplot(data=train_df, x="class", y=feature, ax=axis, palette="Set2")
    axis.set_title(feature)
    axis.set_xlabel("Class")
    axis.set_ylabel("Value")

for axis in axes[len(top_boxplot_features):]:
    axis.axis("off")

boxplot_path = save_current_figure("top_feature_boxplots.png")
plt.show()

print(f"Saved figure: {boxplot_path}")

## 9. Explore the geometry of the dataset with PCA

In [ ]:
scaler_for_pca = StandardScaler()
X_scaled_full = scaler_for_pca.fit_transform(X)

pca_full = PCA(random_state=RANDOM_STATE)
pca_full.fit(X_scaled_full)
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(10, 5))
plt.plot(
    np.arange(1, len(cumulative_variance) + 1),
    cumulative_variance,
    marker="o",
    linewidth=2,
)
plt.axhline(0.80, color="tomato", linestyle="--", label="80% explained variance")
plt.axhline(0.90, color="darkgreen", linestyle="--", label="90% explained variance")
plt.title("Cumulative explained variance from PCA")
plt.xlabel("Number of principal components")
plt.ylabel("Cumulative explained variance")
plt.legend()
variance_path = save_current_figure("pca_cumulative_variance.png")
plt.show()

print(f"Saved figure: {variance_path}")
print("Components needed for 80% variance:", int(np.argmax(cumulative_variance >= 0.80) + 1))
print("Components needed for 90% variance:", int(np.argmax(cumulative_variance >= 0.90) + 1))

pca_2d = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca_2d = pca_2d.fit_transform(X_scaled_full)
pca_plot_df = pd.DataFrame(X_pca_2d, columns=["PC1", "PC2"]).assign(class_label=y.values)

plt.figure(figsize=(10, 7))
sns.scatterplot(
    data=pca_plot_df.sample(min(4000, len(pca_plot_df)), random_state=RANDOM_STATE),
    x="PC1",
    y="PC2",
    hue="class_label",
    palette="Set1",
    alpha=0.7,
    s=55,
)
plt.title("PCA projection of the training set")
plt.xlabel(f"PC1 ({pca_2d.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"PC2 ({pca_2d.explained_variance_ratio_[1]:.1%} variance)")
pca_scatter_path = save_current_figure("pca_scatter.png")
plt.show()

print(f"Saved figure: {pca_scatter_path}")

## 10. Create training and validation splits

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_STATE,
)

print("Training split:", X_train.shape, y_train.shape)
print("Validation split:", X_valid.shape, y_valid.shape)

## 11. Fit the model with the fixed best hyperparameters

In [ ]:
print("Fixed best hyperparameters:")
print(BEST_PARAMS)

fitted_model = MODEL_PIPELINE
fitted_model.fit(X_train, y_train)

print("Model fitted successfully.")

## 12. Evaluate the model on the validation split

In [ ]:
valid_predictions = fitted_model.predict(X_valid)
validation_report = evaluate_predictions(y_valid, valid_predictions)
validation_accuracy = validation_report["accuracy"]

print("Validation accuracy:", round(validation_accuracy, 4))
print("Confusion matrix:")
print(np.array(validation_report["confusion_matrix"]))

disp = ConfusionMatrixDisplay.from_predictions(
    y_valid,
    valid_predictions,
    display_labels=["Undamaged (0)", "Damaged (1)"],
    cmap="Blues",
    colorbar=False,
)
disp.ax_.set_title("Validation confusion matrix")
confusion_path = save_current_figure("validation_confusion_matrix.png")
plt.show()

print(f"Saved figure: {confusion_path}")

## 13. Interpret the fitted model

In [ ]:
coefficients = pd.Series(
    fitted_model.named_steps["model"].coef_.ravel(),
    index=X.columns,
)

coefficient_plot = pd.concat(
    [
        coefficients.sort_values().head(10),
        coefficients.sort_values(ascending=False).head(10),
    ]
)

plt.figure(figsize=(12, 8))
sns.barplot(x=coefficient_plot.values, y=coefficient_plot.index, palette="coolwarm")
plt.title("Most influential standardized coefficients")
plt.xlabel("Coefficient value")
plt.ylabel("Predictor")
coef_path = save_current_figure("logreg_top_coefficients.png")
plt.show()

print(f"Saved figure: {coef_path}")

## Interpretation

Logistic regression provides a linear decision boundary.
It is easy to interpret because the sign and magnitude of the coefficients indicate how each standardized predictor moves the decision toward class 0 or class 1.

## 14. Refit on the full training set and generate submission

In [ ]:
final_model = MODEL_PIPELINE
final_model.fit(X, y)
test_predictions = final_model.predict(X_test_kaggle)

submission_df = pd.DataFrame(
    {
        "id": test_df["id"],
        "class": test_predictions.astype(int),
    }
)

submission_path = submission_dir / f"{NOTEBOOK_SLUG}_submission.csv"
submission_df.to_csv(submission_path, index=False)
submission_df.head()

print(f"Submission saved to: {submission_path}")

## 15. Save a compact experiment summary

In [ ]:
summary_payload = {
    "model_name": MODEL_NAME,
    "mode": "fixed_best_params",
    "validation_accuracy": validation_accuracy,
    "best_params": BEST_PARAMS,
    "output_dir": str(output_dir),
    "submission_path": str(submission_path),
}

summary_path = output_dir / "summary.json"
summary_path.write_text(json.dumps(summary_payload, indent=2))

print("Summary saved to:", summary_path)
summary_payload